# How the Coster warehouse works, according to its data

This notebook reads the files Coster shared and works out, from the data, the rules the automated warehouse follows:
how capacity is counted, where a load unit is stored, what fills and frees a cell, how fast cells run out, and how far ahead the data lets us see.
It ends with a few data-based notes for designing a forecast.

**Data:** the nine Excel files Coster sent on 23 Oct 2026, kept outside the repo in the project's `data` folder.

**Contents**
1. The data
2. How capacity is counted
3. Where a load unit is stored
4. What fills a cell and what frees it
5. How fast cells run out
6. How far ahead we can see
7. The rules in one table
8. Notes for the forecast
9. What we need from Coster

## Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", "{:,.1f}".format)
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.color": "#e6e6e6",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.axisbelow": True,
    }
)

DATA = Path(os.environ.get("COSTER_DATA_DIR", "../../data"))  # outside the repo
OUT = Path("../outputs/audit")  # git-ignored
OUT.mkdir(parents=True, exist_ok=True)
FILE_DATE = pd.Timestamp("2026-10-23")  # date in the file names

A **load unit (UDC, *unità di carico*)** is what one cell of the warehouse holds. The load unit types in the rules file are pallets, pallboxes and containers (section 3).

The files use Italian names. In our tables and charts we use the matching English category; the data itself is not changed.
Cell classes: *bassa / media / alta* are low / medium / high. *Piena / vuota* are full / empty.
Quality codes L / B / S are available / blocked / suspended, as defined by Coster.

In [ ]:
CLASSES = {
    "Cella Bassa": "Low",
    "Cella Media": "Medium",
    "Cella Alta": "High",
    "Cella Media (L1000)": "Medium L1000",
    "Cella Alta (L1000)": "High L1000",
}
ORDER = list(CLASSES.values())
QUALITY = {"L": "available", "B": "blocked", "S": "suspended"}

## 1. The data

Empty values are written as the text `[NULL]` and some names have extra spaces. Every file is read the same way:
`[NULL]` becomes empty, spaces are removed, and codes stay as text so leading zeros are kept.

In [ ]:
def read(name, sheet=0, text=(), dates=None):
    """Read one sheet: [NULL] -> empty, spaces stripped, given columns parsed as dates."""
    df = pd.read_excel(
        DATA / name, sheet_name=sheet, na_values=["[NULL]"], dtype={c: str for c in text}
    )
    df.columns = df.columns.str.strip()
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].str.strip()
    for c, fmt in (dates or {}).items():
        df[c] = pd.to_datetime(df[c].astype(str), format=fmt) if fmt else pd.to_datetime(df[c])
    if "TP_CELLA_DESC" in df:
        df["class"] = df["TP_CELLA_DESC"].map(CLASSES)
    return df


ymd = "%Y%m%d"
plan = read(
    "20261023 - 1 - PU - PRODUZIONE OdL_OdV.xlsx",
    text=["OdV", "OdL", "ARTICOLO"],
    dates={"DT_INIZIO_OdL": ymd, "DT_FINE_OdL": ymd, "DT_USCITA_MERCE": ymd},
)
versamento = read(
    "20261023 - 2 - PU - VERSAMENTO UDC.xlsx", text=["OdL"], dates={"DT_REGISTRAZIONE": ymd}
)
rilascio = read(
    "20261023 - 3 - PU - RILASCIO UDC.xlsx", text=["OdL"], dates={"DT_REGISTRAZIONE": ymd}
)
snap = read(
    "20261023 - 4 - PU - SATURAZIONE CELLE MAGAZZINO.xlsx",
    "Foglio1",
    text=["ID_CELLA", "ID_UDC", "ARTICOLO", "ODV"],
    dates={"DT_UM": None, "DT_ESTRAZIONE": None},
)
snap_summary = read("20261023 - 4 - PU - SATURAZIONE CELLE MAGAZZINO.xlsx", "Foglio2")
hist = read(
    "20261023 - 5 - PU - ANALISI STORICA SATURAZIONE CELLE.xlsx", dates={"DT_ANALISI": None}
)
picking = read("20261023 - 6 - PU - ANALISI STORICA PICKING.xlsx", dates={"GIORNO": None})
inbound = read("B01_B02 TO WAREHOUSE PER DAY.xlsx", dates={"DT_CREAZIONE": "%d/%m/%Y"})
machines = read("PCS-HOUR.xlsx")
rules = read("UDC vs CELLE.xlsx")

In [ ]:
# The raw files, before cleaning: how often [NULL] and extra spaces appear
raw_counts = []
for name in sorted(p.name for p in DATA.glob("*.xlsx") if not p.name.startswith("~$")):
    for sheet, raw in pd.read_excel(DATA / name, sheet_name=None, dtype=str).items():
        raw_counts.append(
            (
                name,
                sheet,
                int((raw == "[NULL]").sum().sum()),
                sum(c != c.strip() for c in raw.columns),
                int(sum((raw[c].dropna() != raw[c].dropna().str.strip()).sum() for c in raw)),
            )
        )
pd.DataFrame(
    raw_counts,
    columns=[
        "file",
        "sheet",
        "cells written as [NULL]",
        "column names with extra spaces",
        "values with extra spaces",
    ],
).set_index(["file", "sheet"])

In [ ]:
def period(dates):
    first, last = dates.min(), dates.max()
    return f"{first:%d %b %Y}" if first == last else f"{first:%d %b %Y} to {last:%d %b %Y}"


files = [
    (
        "Production plan",
        plan,
        "daily",
        "work orders planned for the next three weeks",
        "DT_FINE_OdL",
    ),
    ("Versamento", versamento, "daily", "load units deposited, previous day", "DT_REGISTRAZIONE"),
    ("Rilascio", rilascio, "daily", "load units released, previous day", "DT_REGISTRAZIONE"),
    (
        "Snapshot (cells)",
        snap,
        "daily",
        "every cell today, with the load unit in it",
        "DT_ESTRAZIONE",
    ),
    ("Snapshot (summary)", snap_summary, "daily", "cell counts by class and status", None),
    (
        "Occupancy history",
        hist,
        "daily",
        "total, occupied and free cells per day, area, class",
        "DT_ANALISI",
    ),
    (
        "Picking history",
        picking,
        "daily",
        "pallets used for stamping, load units deleted",
        "GIORNO",
    ),
    (
        "Inbound B01/B02",
        inbound,
        "one-off",
        "approximate load units coming in per day and class",
        "DT_CREAZIONE",
    ),
    ("Machine speeds", machines, "one-off", "pieces per hour for each machine", None),
    (
        "UDC vs cells",
        rules,
        "one-off",
        "cell classes each load unit type may use, by priority",
        None,
    ),
]
overview = pd.DataFrame(
    [(name, freq, what, len(df), period(df[d]) if d else "") for name, df, freq, what, d in files],
    columns=["file", "sent (Coster)", "content (Coster)", "rows", "period"],
).set_index("file")
overview

Only the occupancy history, the picking history and the inbound file go back in time. Versamento, rilascio and the snapshot cover one day, the plan is a single version, and the machine speeds and the rules have no dates.

A few quality points to keep in mind when reading the rest:

In [ ]:
per_day = hist.pivot_table(index="DT_ANALISI", columns=["COD_AREA", "TP_CELLA"], values="OCCUPATE")
repeat_days = per_day.index[per_day.diff().abs().sum(axis=1) == 0]
all_days = pd.date_range(hist.DT_ANALISI.min(), hist.DT_ANALISI.max())
pd.DataFrame(
    [
        (
            "Occupancy history",
            "occupied + free = total",
            f"{(hist.OCCUPATE + hist.LIBERE != hist.TOTALI).sum()} rows break it",
        ),
        (
            "Occupancy history",
            "days with no rows",
            ", ".join(all_days.difference(hist.DT_ANALISI.unique()).strftime("%d %b")),
        ),
        (
            "Occupancy history",
            "days whose record is identical to the previous day",
            f"{len(repeat_days)} days, mostly on "
            + " and ".join(
                d + "s" for d in pd.Series(repeat_days.day_name()).value_counts().index[:2]
            ),
        ),
        (
            "Production plan",
            "expected dispatch before the end of production",
            f"{(plan.DT_USCITA_MERCE < plan.DT_FINE_OdL).sum()} of {len(plan)} work orders",
        ),
        (
            "Picking history",
            "days without a value for load units deleted",
            str(int(picking.UDC_CANCELLATE_PROD.isna().sum())),
        ),
        (
            "All files",
            "dates with a time other than 00:00",
            str(
                sum(
                    int((df[c] != df[c].dt.normalize()).sum())
                    for df in (plan, versamento, rilascio, snap, hist, picking, inbound)
                    for c in df.select_dtypes(include="datetime").columns
                )
            ),
        ),
    ],
    columns=["file", "check", "result"],
).set_index(["file", "check"])

## 2. How capacity is counted

**One cell holds at most one load unit.** In the snapshot, no load unit number appears in two cells, and no empty cell has one. A few full cells have no load unit number:

In [ ]:
full = snap["ST_CELLA_DESC"] == "PIENA"
print(
    "full cells:",
    int(full.sum()),
    "| with a load unit:",
    int(snap.loc[full, "ID_UDC"].notna().sum()),
    "| different load units:",
    snap.loc[full, "ID_UDC"].nunique(),
    "| empty cells with a load unit:",
    int(snap.loc[~full, "ID_UDC"].notna().sum()),
)

**Capacity is the number of usable cells per class.** The two areas (C01, C02) have the same number of cells in every class. Disabled cells (status 5) do not count: compare the last two columns.

In [ ]:
daily = hist.groupby(["DT_ANALISI", "class"])[["TOTALI", "OCCUPATE", "LIBERE"]].sum()
cap = daily["TOTALI"].unstack()[ORDER]
occ = daily["OCCUPATE"].unstack()[ORDER]
free = daily["LIBERE"].unstack()[ORDER]

snap["area"] = snap.ID_CELLA.str[:3]
by_class = snap.groupby("class")
pd.DataFrame(
    {
        "cells in C01": snap[snap.area == "C01"].groupby("class").size(),
        "cells in C02": snap[snap.area == "C02"].groupby("class").size(),
        "disabled": by_class.ST_ATT.apply(lambda s: (s == 5).sum()),
        "snapshot cells minus disabled": by_class.size()
        - by_class.ST_ATT.apply(lambda s: (s == 5).sum()),
        f"capacity in history ({cap.index[-1]:%d %b})": cap.iloc[-1],
    }
).loc[ORDER].astype(int)

**A full cell counts as full whatever is in it.** Some full cells hold a quantity of 0, a few have no load unit number:

In [ ]:
print(
    "full cells with quantity 0:",
    int((full & (snap.QTA == 0)).sum()),
    "| full cells without a load unit number:",
    int((full & snap.ID_UDC.isna()).sum()),
)

**Capacity changes over time.** In summer the number of cells per class changed: High lost cells, Low and Medium gained cells.

In [ ]:
fig, axes = plt.subplots(len(ORDER), 1, figsize=(11, 9), sharex=True)
for ax, c in zip(axes, ORDER, strict=True):
    ax.step(cap.index, cap[c], where="post", color="grey")
    ax.step(occ.index, occ[c], where="post", color="tab:blue")
    ax.set_ylim(0, cap[c].max() * 1.1)
    ax.set_ylabel("cells")
    ax.set_title(c, fontsize=10)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.suptitle("Occupied cells (blue) and capacity (grey) per class, 2026", x=0.01, ha="left")
plt.tight_layout()
plt.savefig(OUT / "occupancy.png")
plt.show()

changed = cap.diff().fillna(0).ne(0).any(axis=1)
print("Capacity changed on:", ", ".join(cap.index[changed].strftime("%d %b")))
pd.DataFrame({"1 Jul": cap.loc["2026-07-01"], "22 Oct": cap.iloc[-1]}).astype(int).T

Because the number of cells changes, a fill rate mixes changes in stock with changes in capacity. The rest of the notebook works with cell counts.

## 3. Where a load unit is stored

The rules file lists, for every load unit type, the cell classes it may use and in which order (priority 1 is the first choice).
The load unit types fall into a few patterns:

In [ ]:
prio = rules.pivot_table(index="TP_UDC", columns="class", values="PRIORITA_UTILIZZO")[ORDER]
desc = rules.drop_duplicates("TP_UDC").set_index("TP_UDC").TP_UDC_DESC
height = desc.str.extract(r"h\.\s*([\d.]+)", expand=False).str.replace(".", "", regex=False)
wide = desc.str.contains("1.000 x", regex=False)
patterns = (
    prio.fillna(0)
    .astype(int)
    .astype(str)
    .replace("0", "-")
    .assign(example=desc, height=height, wide=wide)
    .groupby(ORDER)
    .agg(
        **{
            "load unit types": ("example", "size"),
            "heights (mm)": ("height", lambda h: ", ".join(sorted(set(h), key=int))),
            "1000-wide types": ("wide", "sum"),
            "example": ("example", "first"),
        }
    )
    .sort_values("load unit types", ascending=False)
    .reset_index()
)
print(f"{prio.shape[0]} load unit types, {len(patterns)} patterns")
patterns

Each column is the priority of that class for the pattern, "-" means not allowed.
The pattern follows the size of the load unit (heights column): the lowest units may use every class and start at Low, medium-height units start at Medium, the tallest start at High, and the patterns made only of 1000-wide types use only the two L1000 classes.

**Does the warehouse follow this order?** For each stored load unit we look up the priority of the cell it sits in:

In [ ]:
stored = snap[full & snap.TP_UDC.notna()].astype({"TP_UDC": int})
placed = stored.merge(
    rules[["TP_UDC", "TP_CELLA_DESC", "PRIORITA_UTILIZZO"]],
    on=["TP_UDC", "TP_CELLA_DESC"],
    how="left",
)
by_priority = pd.crosstab(placed["class"], placed.PRIORITA_UTILIZZO.fillna(0).astype(int))
by_priority.columns = [f"priority {p}" if p else "not allowed" for p in by_priority.columns]
first_choice = rules[rules.PRIORITA_UTILIZZO == 1].set_index("TP_UDC")["class"]
overflow_high = placed[(placed["class"] == "High") & (placed.PRIORITA_UTILIZZO > 1)]
print(f"In their first-choice class: {(placed.PRIORITA_UTILIZZO == 1).mean() * 100:.0f} %")
print(
    "Overflow load units in High cells, by first-choice class:",
    overflow_high.TP_UDC.map(first_choice).value_counts().to_dict(),
)
by_priority.loc[ORDER]

On the snapshot day no load unit sits in a class it is not allowed in, and every overflow unit sits in a later priority. Most of the overflow is Medium-type pallets in High cells.
The data has no placement times, so we cannot see whether the first choice was full when an overflow unit arrived.

## 4. What fills a cell and what frees it

**Inflow.** B01/B02 gives an approximate count of load units coming in each day, by class.

In [ ]:
inflow = inbound.pivot_table(index="DT_CREAZIONE", columns="class", values="N_UDC", aggfunc="sum")
inflow = inflow.reindex(pd.date_range(inflow.index.min(), inflow.index.max())).fillna(0)[ORDER]

weekly = inflow.resample("W-MON").sum()
fig, ax = plt.subplots(figsize=(11, 3.5))
bottom = np.zeros(len(weekly))
for c in ORDER:
    ax.bar(weekly.index, weekly[c], bottom=bottom, width=5, label=c)
    bottom += weekly[c].to_numpy()
ax.set(
    title="Load units coming in per week, by class (B01/B02)",
    xlabel="week",
    ylabel="load units per week",
)
ax.set_ylim(0, weekly.sum(axis=1).max() * 1.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.legend(ncol=5, fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "inflow_weekly.png")
plt.show()

**Outflow.** No file gives outflow by class. The picking file gives, for the whole warehouse, pallets used for stamping and load units deleted for production.
Both sides follow the working week:

In [ ]:
pick = picking.set_index("GIORNO")
by_weekday = pd.DataFrame(
    {
        "load units in": inflow.sum(axis=1).groupby(inflow.index.day_of_week).mean(),
        "pallets used for stamping": pick.PALLET_STAMP.groupby(pick.index.day_of_week).mean(),
        "load units deleted": pick.UDC_CANCELLATE_PROD.groupby(pick.index.day_of_week).mean(),
    }
)
by_weekday.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
ax = by_weekday.plot.bar(figsize=(9, 3.2), rot=0, width=0.8)
ax.set(title="Average per day of the week", xlabel="", ylabel="count per day")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT / "weekday_profile.png")
plt.show()

**Do these flows explain the change in stock?** By definition, the occupied cells follow a balance: *occupied today = occupied yesterday + in - out*.
We take the deleted load units as the outflow (our assumption) and fit a straight line between the daily change in occupied cells and (a) the weekday, (b) the observed flows, (c) both. The table gives the share of the change explained (R², %).

In [ ]:
def r2(y, X):
    """Share of the variance of y explained by a least-squares line on X, in %."""
    d = pd.concat([y.rename("y"), X], axis=1).dropna()
    A = np.column_stack([np.ones(len(d)), d.drop(columns="y").to_numpy(float)])
    coef, *_ = np.linalg.lstsq(A, d["y"].to_numpy(float), rcond=None)
    rest = d["y"] - A @ coef
    return 100 * (1 - (rest**2).sum() / ((d["y"] - d["y"].mean()) ** 2).sum())


occ_days = occ.reindex(pd.date_range(occ.index.min(), occ.index.max()))
change = occ_days.diff()
weekday = (
    pd.get_dummies(change.index.day_of_week, drop_first=True).set_index(change.index).astype(float)
)
flows = pd.DataFrame({"in": inflow.sum(axis=1), "deleted": pick.UDC_CANCELLATE_PROD}).reindex(
    change.index
)
total = change.sum(axis=1, min_count=len(ORDER))
rows = [
    (
        "whole warehouse",
        r2(total, weekday),
        r2(total, flows),
        r2(total, pd.concat([weekday, flows], axis=1)),
    )
]
for c in ORDER:
    x = inflow[c].reindex(change.index).rename("in").to_frame()
    rows.append(
        (
            c,
            r2(change[c], weekday),
            r2(change[c], x),
            r2(change[c], pd.concat([weekday, x], axis=1)),
        )
    )
explained = pd.DataFrame(
    rows, columns=["daily change in occupied cells", "weekday", "observed flows", "both"]
)
explained = explained.set_index("daily change in occupied cells")
explained["not explained"] = 100 - explained["both"]
explained.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(flows["in"] - flows["deleted"], total, s=12, alpha=0.6)
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
ax.set(
    title="Daily change in occupied cells vs observed flows",
    xlabel="load units in minus load units deleted (per day)",
    ylabel="change in occupied cells (per day)",
)
plt.tight_layout()
plt.savefig(OUT / "balance_scatter.png")
plt.show()

Most of the daily change is not explained by the weekday or by the observed flows. The chart shows the same for the whole warehouse: each dot is a day.

**Moves in progress and quality status.** Cells carry the number of pallet moves in and out under way, and stored load units have a quality status:

In [ ]:
moves = pd.crosstab(
    [snap.NR_MISS_IN.rename("moves in"), snap.NR_MISS_OUT.rename("moves out")],
    snap.ST_CELLA_DESC.map({"PIENA": "full", "VUOTA": "empty"}).rename("cell"),
)
display(moves)
quality = snap.loc[full, "ST_QUALITA"].map(QUALITY).fillna("no code").rename("quality status")
pd.crosstab(snap.loc[full, "class"], quality).loc[ORDER]

Some full cells have a move in, and some empty cells a move out. The data does not say whether a cell turns full when a move is created or when the pallet arrives.
Blocked and suspended load units still take up their cell.

## 5. How fast cells run out

The largest drops in free cells from one daily record to the next, and how often each class reached 0 free cells:

In [ ]:
zero = free == 0
starts = zero & ~zero.shift(1, fill_value=False)
runs = zero.cumsum() - zero.cumsum().where(~zero).ffill().fillna(0)
pd.DataFrame(
    {
        "largest drop in 1 day": -free.diff().min(),
        "largest drop in 2 days": -free.diff(2).min(),
        "days with 0 free": zero.sum(),
        "periods with 0 free": starts.sum(),
        "longest period (days)": runs.max(),
    }
).loc[ORDER].astype(int)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6))
for i, c in enumerate(ORDER):
    ax.scatter(free.index[zero[c]], [i] * zero[c].sum(), marker="|", s=120)
ax.set_yticks(range(len(ORDER)), ORDER)
ax.set(title="Days with 0 free cells, per class", xlabel="2026")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
plt.tight_layout()
plt.savefig(OUT / "zero_free_days.png")
plt.show()

Free cells can drop sharply from one daily record to the next (first two columns). Medium has the most days and the longest periods at 0 free cells; High and Medium L1000 follow, with shorter periods.

## 6. How far ahead we can see

What each file tells us on the day it arrives, compared with the file date:

In [ ]:
latest = {
    "Production plan": plan.DT_INIZIO_OdL.min(),
    "Versamento": versamento.DT_REGISTRAZIONE.max(),
    "Rilascio": rilascio.DT_REGISTRAZIONE.max(),
    "Snapshot (cells)": snap.DT_ESTRAZIONE.max(),
    "Occupancy history": hist.DT_ANALISI.max(),
    "Picking history": pick.index.max(),
    "Inbound B01/B02": inbound.DT_CREAZIONE.max(),
}
seen = {
    k: (
        "from the file date on"
        if k == "Production plan"
        else "same day"
        if d == FILE_DATE
        else f"{(FILE_DATE - d).days} day(s) back"
    )
    for k, d in latest.items()
}
pd.DataFrame({"sent (Coster)": overview["sent (Coster)"], "covers": seen}).loc[list(latest)]

In [ ]:
ahead = (plan.DT_FINE_OdL.max() - FILE_DATE).days
gap = (plan.DT_USCITA_MERCE - plan.DT_FINE_OdL).dt.days
print(f"The plan shows production up to {ahead} days after the file date.")
bands = pd.cut(
    gap,
    [-999, -1, 0, 3, 7, 999],
    labels=[
        "before end of production",
        "same day",
        "1 to 3 days later",
        "4 to 7 days later",
        "more than 7 days later",
    ],
)
ax = bands.value_counts(sort=False).plot.barh(figsize=(8, 2.8), color="tab:blue")
ax.invert_yaxis()
ax.set(title="Expected dispatch vs planned end of production", xlabel="work orders", ylabel="")
plt.tight_layout()
plt.savefig(OUT / "dispatch_vs_end.png")
plt.show()

Production is known for the coming weeks, but only by day. Dispatch is known only as an expected date, and for part of the work orders that date comes before production ends.
Everything recorded (deposits, releases, occupancy) arrives one day late. No date in any file has a time other than 00:00 (section 1), so no file says at what hour something happened.

## 7. The rules in one table

In [ ]:
share_first = (placed.PRIORITA_UTILIZZO == 1).mean() * 100
not_allowed = int(placed.PRIORITA_UTILIZZO.isna().sum())
n_units, n_full = snap.loc[full, "ID_UDC"].nunique(), int(snap.loc[full, "ID_UDC"].notna().sum())
not_explained = explained.loc["whole warehouse", "not explained"]
is_weekend = lambda idx: idx.day_of_week >= 5  # noqa: E731
weekend_in = inflow.sum(axis=1)[is_weekend(inflow.index)].sum() / inflow.values.sum() * 100
weekend_del = (
    pick.UDC_CANCELLATE_PROD[is_weekend(pick.index)].sum() / pick.UDC_CANCELLATE_PROD.sum() * 100
)
rows_per_day = hist.groupby("DT_ANALISI").size()
pd.DataFrame(
    [
        (
            "One cell holds at most one load unit",
            "shown",
            f"{n_units} different load unit numbers in {n_full} full cells; "
            f"{int(full.sum()) - n_full} full cells have no number",
        ),
        (
            "Capacity = usable cells per class; disabled cells do not count",
            "shown",
            f"on {cap.index[-1]:%d %b}, history totals equal snapshot cells minus disabled cells",
        ),
        ("Capacity per class can change", "shown", f"changed on {int(changed.sum())} days in 2026"),
        (
            "A load unit goes to the first class in its priority order that has room",
            "fits the data",
            f"{share_first:.0f} % in first-choice class, {not_allowed} in a class not allowed; "
            "no placement times",
        ),
        (
            "High holds Medium-type pallets",
            "shown",
            f"{len(overflow_high)} overflow load units in High on the snapshot day",
        ),
        (
            "...and it does so because Medium is full",
            "open",
            f"Medium had {int(free['Medium'].iloc[-1])} free cells on the last history day; "
            "no placement times",
        ),
        (
            "Retrieval is first in, first out",
            "open",
            "no record of when a load unit entered or left",
        ),
        (
            "Inflow and outflow follow the working week",
            "shown",
            f"{weekend_in:.0f} % of inflow and {weekend_del:.0f} % of deleted load units "
            "fall on weekends",
        ),
        (
            "The observed flows explain the change in stock",
            "no",
            f"{not_explained:.0f} % of the daily change is not explained",
        ),
        (
            "Blocked and suspended load units keep their cell",
            "shown",
            f"{int(snap.loc[full, 'ST_QUALITA'].isin(['B', 'S']).sum())} stored",
        ),
        (
            "Production is known in advance, by day",
            "shown",
            f"{ahead} days ahead, one plan version",
        ),
        ("Dispatch is known in advance", "open", "expected dates only, no actual dispatches"),
        (
            "Occupancy is recorded once a day",
            "shown",
            f"one set of rows per date ({rows_per_day.mode().iloc[0]} rows on most dates), "
            "no hours; "
            f"on {len(repeat_days)} days the record is identical to the previous day",
        ),
    ],
    columns=["rule", "status", "evidence"],
).set_index("rule")

## 8. Notes for the forecast

These follow from the data above.

**What to forecast.** Free cells per class per day (the `LIBERE` count), not fill rates, because capacity changes (section 2). Every class reached 0 free cells at least once; Medium did so most often (section 5).

**The bar to beat.** The simplest forecast says "tomorrow is like today". Its average error in cells, one and two days ahead:

In [ ]:
free_days = free.reindex(pd.date_range(free.index.min(), free.index.max()))
baseline = pd.DataFrame(
    {
        "1 day ahead": free_days.diff().abs().mean(),
        "2 days ahead": free_days.diff(2).abs().mean(),
        "average free cells": free_days.mean(),
    }
).loc[ORDER]
baseline

**Inputs on the day of a forecast.** Today's snapshot, the occupancy history up to yesterday, the picking history, and the plan as issued that day.
The weekday and the observed flows explain little of the daily change (section 4). Whether the plan explains more cannot be tested yet, because there is only one plan version.

**Training data.** Capacity changed during the summer (section 2), so cell counts before and after differ; days whose record is identical to the previous day should be marked (section 1); tests should use later days than the training data.
To train with the plan, we need past versions of it, as they were on each day. Keeping every daily file from now on builds that history, and two consecutive snapshots show which load units moved.

## 9. What we need from Coster

1. Plans and snapshots from earlier months, if they are kept.
2. Actual dispatches, with date and time.
3. The time at which each daily file is extracted, and the time zone.
4. Inflow and outflow split by cell class.
5. Why some days are missing from the history, and why on some days the record is identical to the previous day.

**Assumptions used**
- Dates are local plant dates; no file states a time zone.
- The two areas are added together when we speak of a cell class.
- The plan in the 23 Oct file is the plan as issued on that day.
- A deleted load unit no longer occupies a cell (section 4).